# Exploratory Data Analysis Organized

In [1]:
print('Kernel test 123')

Kernel test 123


In [ ]:
# Step 0, lets import the regular stuff we will probably need
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
summarize)
print('Done part 1212323')

# MORE MOUSE BITES
# Ehh, we should really start learning SCIKITLEARN next or whatever its called
# but I think for this project its not horrible to stick with the ISLP resources
# since we ARE straying from the notes now
from ISLP import confusion_table
from ISLP.models import contrast
from sklearn.discriminant_analysis import \
(LinearDiscriminantAnalysis as LDA ,
QuadraticDiscriminantAnalysis as QDA)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
print('Done part 2 with git')

Done part 1212323
Done part 2 with git


In [32]:
# Import the data:
Titanic_train = pd.read_csv('titanic_data/train.csv')

Titanic_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Here is the summary of our EDA.

<img src="images_titanic/EDA summary.png" width="1300">

The plan is this:

 1(a)Drop 'PassengerId' column

 1(b) Create column called 'Title' to estimate the missing ages. Estimate missing ages

 1(c) Combine SibSp and ParCh columns into one (SibSp + ParCh). Then make it into bins (we dont want bins that are too small, to avoid overfitting). Drop 'SibSP', 'ParCh' columns as well as any created temporarily to calculate this Family_Bins style column

 1(d) Make 'Has_Cabin_Record' column. Drop 'Cabin' column 

 1(e) Use ticket to calculate log+1(Fare). Create 'log_plus1_Fare' column. Drop 'Fare' and 'Ticket' columns.

 1(f) Delete the rows where 'Embarked' is null. Setup so that for future training, we assume Embarked = Mode (S?)

In [33]:
#1(a)

# We drop the useless looking column
Titanic_train = Titanic_train.drop(columns='PassengerId')

Titanic_train.head()

print('hello')

hello


In [34]:
# 1(b)

print('The median age  was ' + str(Titanic_train['Age'].median()))

# Then we use regex to find the different groups, and make a new column. The column we care about is 'Name', we categorize according to cat we found and what google says they mean
conditions_name = [
    Titanic_train['Name'].str.contains(r'Master') 
    , Titanic_train['Name'].str.contains(r'Mrs.') | Titanic_train['Name'].str.contains(r'Mme.')# Married woman Mme
    , Titanic_train['Name'].str.contains(r'Mr.') # Mr.
    , Titanic_train['Name'].str.contains(r'Miss.') | Titanic_train['Name'].str.contains(r'Mlle.') # Girl or unmarried woman, Unmarried
]

choices_name = ['Master'
                , 'Mrs/Mme'
                , 'Mr'
                , 'Miss/Mlle']

Titanic_train['Title'] = np.select(conditions_name, choices_name, default = 'Other')

# We review the value counts, Other is a BIT suspect in terms of being too small a category, but since we are using this for age estimation as opposed to category, I think this is ok
Titanic_train['Title'].value_counts()

Titanic_train['Age'] = Titanic_train['Age'].fillna(
    Titanic_train.groupby('Title')['Age'].transform('median')
)

print('The median age has changed to be ' + str(Titanic_train['Age'].median()))

The median age  was 28.0
The median age has changed to be 30.0


In [35]:
# 1(c)
Titanic_train['Family'] = Titanic_train['SibSp']+Titanic_train['Parch']

# Create bins for SibSp and ParCh
bins_01_2 = [-0.5, 0.5, 1.5, 1000]
labels_01_2 = ['Zero', 'One', 'Two or more']

# Create bins for "Family"
bins_012_3 = [-0.5, 0.5, 1.5, 2.5, 1000]
labels_012_3 = ['Zero', 'One', 'Two', 'Three or more']

Titanic_train['Sibsp_bins'] = pd.cut(Titanic_train['SibSp'], bins_01_2, labels = labels_01_2)
Titanic_train['Parch_bins'] = pd.cut(Titanic_train['Parch'], bins_01_2, labels = labels_01_2)
Titanic_train['Family_bins'] = pd.cut(Titanic_train['Family'], bins_012_3, labels = labels_012_3)
Titanic_train.head(10)

# Drop actual values. Bins to be decided later
Titanic_train = Titanic_train.drop(columns='SibSp')
Titanic_train = Titanic_train.drop(columns='Parch')
Titanic_train = Titanic_train.drop(columns='Family')

In [36]:
# 1(d)
Titanic_train['Has_Cabin_Record'] = Titanic_train['Cabin'].notna().astype(int)

Titanic_train = Titanic_train.drop(columns='Cabin')

In [37]:
# 1(e)

ticket_counts = Titanic_train['Ticket'].value_counts()
group_size_by_ticket = Titanic_train['Ticket'].map(ticket_counts)

Titanic_train['Individual_Fare_LogPlus1'] = np.log1p(Titanic_train['Fare'] / group_size_by_ticket)

Titanic_train = Titanic_train.drop(columns='Fare')
Titanic_train = Titanic_train.drop(columns='Ticket')

In [38]:
# 1(f)
Titanic_train = Titanic_train.dropna(subset=['Embarked'])

# 1(g) (For test data:) Fill missing 'Embarked' values in the test set with 'S'
#Titanic_test['Embarked'] = Titanic_test['Embarked'].fillna('S')

In [39]:
# 1(g) 
Titanic_train = Titanic_train.drop(columns='Name')
Titanic_train = Titanic_train.drop(columns='Title')

In [56]:
# 1(h) 
# Change the data type of some of the columns to the 'category' data type to make them easier to manipulate.
Titanic_train['Sex'] = Titanic_train['Sex'].astype('category')
Titanic_train['Embarked'] = Titanic_train['Embarked'].astype('category')

# Force Family_bins to be an UNORDERED category so MS() generates dummy variables. Thank you Gemini
Titanic_train['Family_bins'] = pd.Categorical(
    Titanic_train['Family_bins'], 
    categories=['Zero', 'One', 'Two', 'Three or more'], 
    ordered=False
)

# Again thank you Gemini for this change in datatype.
Titanic_train['Sibsp_bins'] = pd.Categorical(Titanic_train['Sibsp_bins'], ordered=False)
Titanic_train['Parch_bins'] = pd.Categorical(Titanic_train['Parch_bins'], ordered=False)
Titanic_train['Pclass']=pd.Categorical(Titanic_train['Pclass'], ordered=False)

In [57]:
# Final checkup
Titanic_train

,Survived,Pclass,Sex,Age,Embarked,Sibsp_bins,Parch_bins,Family_bins,Has_Cabin_Record,Individual_Fare_LogPlus1
0,0,3,male,22.0,S,One,Zero,One,0,2.110213
1,1,1,female,38.0,C,One,Zero,One,1,4.280593
2,1,3,female,26.0,S,Zero,Zero,Zero,0,2.188856
3,1,1,female,35.0,S,One,Zero,One,1,3.316003
4,0,3,male,35.0,S,Zero,Zero,Zero,0,2.202765
...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,S,Zero,Zero,Zero,0,2.639057
887,1,1,female,19.0,S,Zero,Zero,Zero,1,3.433987
888,0,3,female,21.0,S,One,Two or more,Three or more,0,2.543569
889,1,1,male,26.0,C,Zero,Zero,Zero,1,3.433987


In [58]:
# So the question becomes 'how do we choose which model to use to train the data'. Well lets list out the possible models that chapter 4 has given us:
# Logistic Regression
# Linear Discriminant Analysis
# Quadratic Discriminant Analysis
# Naive Bayes
# KNN

# Most likely its gonna be QDA but it could be LDA or even Logistic Regression.
# I doubt it will be Naive Bayes due to the number of predictor parameters and the comparative lack of data
# As for Naive Bayes I dont even remember what that one is but it sounds not very flexible. We can look into it.

# So, plan is to fit the model with LR, LDA and QDA. Then investigate Naive Bayes Definition.,

# Then compare model fit using cross validation in 5 groups (the standard)

# Pick the model which has the best Cross-Validation results.

# Fit the data. Kachapaow! Donezo.

# Logistic Regression:

#Titanic_train.describe()

# Titanic train with family column
Titanic_train_f = Titanic_train.copy()
Titanic_train_f  = Titanic_train_f.drop(columns='Sibsp_bins')
Titanic_train_f  = Titanic_train_f.drop(columns='Parch_bins')


# Titanic train with siblings-spouse and parent-child columns
Titanic_train_sp = Titanic_train.copy()
Titanic_train_sp  = Titanic_train_sp.drop(columns='Family_bins')



# Titanic train data for Family version, without the response
Titanic_train_LR_f = Titanic_train_f.copy()
Titanic_train_LR_f  = Titanic_train_f.drop(columns='Survived')

# Titanic train data for SibSpouse-ParChild version, without the response
Titanic_train_LR_sp = Titanic_train_sp.copy()
Titanic_train_LR_sp  = Titanic_train_sp.drop(columns='Survived')


design_LR_f = MS(Titanic_train_LR_f)
design_LR_sp = MS(Titanic_train_LR_sp)

X_LR_f = design_LR_f.fit_transform(Titanic_train_f)
X_LR_sp = design_LR_sp.fit_transform(Titanic_train_sp)

Y_LR_f = Titanic_train_f.Survived == 1
Y_LR_sp = Titanic_train_sp.Survived == 1

glm_LR_f = sm.GLM(Y_LR_f, X_LR_f, family=sm.families.Binomial())
glm_LR_sp = sm.GLM(Y_LR_sp, X_LR_sp, family=sm.families.Binomial())

results_LR_f = glm_LR_f.fit()
results_LR_sp = glm_LR_sp.fit()

summarize(results_LR_f)

#Titanic_train_f

# So we need to convert the following columns:
# Family_bins
# Title
# Embarked
# Sex

,coef,std err,z,P>|z|
intercept,2.4435,0.861,2.839,0.005
Pclass[2],-0.1492,0.416,-0.359,0.720
Pclass[3],-1.2448,0.455,-2.736,0.006
Sex[male],-2.6576,0.206,-12.922,0.000
Age,-0.0403,0.008,-5.029,0.000
Embarked[Q],-0.1320,0.389,-0.339,0.734
Embarked[S],-0.4477,0.242,-1.854,0.064
Family_bins[One],-0.1271,0.252,-0.504,0.615
Family_bins[Two],0.3783,0.290,1.305,0.192
Family_bins[Three or more],-1.1155,0.330,-3.379,0.001


In [60]:
summarize(results_LR_sp)

#Titanic_train['Pclass'].value_counts()

,coef,std err,z,P>|z|
intercept,2.4086,0.868,2.774,0.006
Pclass[2],-0.1365,0.418,-0.326,0.744
Pclass[3],-1.2113,0.456,-2.654,0.008
Sex[male],-2.6188,0.203,-12.885,0.000
Age,-0.0431,0.008,-5.287,0.000
Embarked[Q],-0.0640,0.389,-0.164,0.870
Embarked[S],-0.4230,0.242,-1.746,0.081
Sibsp_bins[One],0.0452,0.224,0.202,0.840
Sibsp_bins[Two or more],-1.1959,0.387,-3.087,0.002
Parch_bins[One],0.2089,0.288,0.724,0.469


In [ ]:
# Now normally I would want to use AIC, BIC or adjusted R^2 (idk eugh), but I 
# havent covered that yet so Im gonna go with something I actually
# understand. So we are going to use K-fold cross validation and compare the uhh
# well we compare somethiing in any case. Durr
# Here is some Gemini code.

#from sklearn.model_selection import cross_val_score
#from sklearn.linear_model import LogisticRegression


# Instantiate with a higher iteration cap
model_f = LogisticRegression(max_iter=1000)
model_sp = LogisticRegression(max_iter=1000)

# Quick comparison using scikit-learn's 5-fold CV
cv_scores_f = cross_val_score(model_f, X_LR_f, Y_LR_f, cv=5, scoring='accuracy')
cv_scores_sp = cross_val_score(model_sp, X_LR_sp, Y_LR_sp, cv=5, scoring='accuracy')

print(f"Family Model CV Accuracy: {cv_scores_f.mean():.4f}")
print(f"SibSp/Parch Model CV Accuracy: {cv_scores_sp.mean():.4f}")

# They are basically equally good, ergo we keep the family one and get rid of SibSp and Parch

Family Model CV Accuracy: 0.8122
SibSp/Parch Model CV Accuracy: 0.8133


In [ ]:
# Now, we are using Titanic_train_f

cv_scores_f

array([0.83146067, 0.80337079, 0.79775281, 0.80337079, 0.82485876])